<a href="https://colab.research.google.com/github/iaolawuyi/keras-text-classification/blob/main/working_with_text_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Text Vectorization Layer

In [1]:
import string

In [2]:
class Vectorizer:

  def standardize(self, text):
    text = text.lower()
    return "".join(char for char in text if char not in string.punctuation)

  def tokenize(self, text):
    text = self.standardize(text)
    return text.split()

  def build_vocabulary(self, dataset):
    self.vocabulary = {
        "": 0,
        "[UNK]": 1
    }
    for text in dataset:
      text = self.standardize(text)
      tokens = self.tokenize(text)
      for token in tokens:
        if token not in self.vocabulary:
          self.vocabulary[token] = len(self.vocabulary)

    self.inverse_vocabulary = dict((value, key) for key, value in self.vocabulary.items())

  def encode(self, text):
    text = self.standardize(text)
    tokens = self.tokenize(text)
    return [self.vocabulary.get(token, 1) for token in tokens]

  def decode(self, int_sequence):
    return " ".join(self.inverse_vocabulary.get(i, "[UNK]") for i in int_sequence)

In [3]:
vectorizer = Vectorizer()
dataset = [
    "I write, erase, rewrite",
    "Erase again, and then",
    "A poppy blooms."
]
vectorizer.build_vocabulary(dataset)

print(vectorizer.vocabulary)

{'': 0, '[UNK]': 1, 'i': 2, 'write': 3, 'erase': 4, 'rewrite': 5, 'again': 6, 'and': 7, 'then': 8, 'a': 9, 'poppy': 10, 'blooms': 11}


In [4]:
test_sentence = "I write, rewrite, and still rewrite again"

encoded_text = vectorizer.encode(test_sentence)
print(encoded_text)

[2, 3, 5, 7, 1, 5, 6]


In [5]:
decoded_text = vectorizer.decode(encoded_text)
print(decoded_text)

i write rewrite and [UNK] rewrite again


In [6]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [7]:
text_vectorization = layers.TextVectorization(output_mode='int')

In [8]:
import re

In [9]:
def custom_standardization_fn(string_tensor):
  lowercase_string = tf.strings.lower(string_tensor)
  return tf.strings.regex_replace(lowercase_string, f"[{re.escape(string.punctuation)}]", "")

def custom_split_fn(string_tensor):
  return tf.strings.split(string_tensor)

text_vectorization = layers.TextVectorization(
    output_mode='int',
    standardize=custom_standardization_fn,
    split=custom_split_fn
)

In [10]:
text_vectorization.adapt(dataset)

In [11]:
vocabulary = text_vectorization.get_vocabulary()
print(vocabulary)

['', '[UNK]', np.str_('erase'), np.str_('write'), np.str_('then'), np.str_('rewrite'), np.str_('poppy'), np.str_('i'), np.str_('blooms'), np.str_('and'), np.str_('again'), np.str_('a')]


In [12]:
encoded_sentence = text_vectorization(test_sentence)
print(encoded_sentence)

tf.Tensor([ 7  3  5  9  1  5 10], shape=(7,), dtype=int64)


In [13]:
inverse_vocab = dict(enumerate(vocabulary))
decoded_sentence = " ".join(inverse_vocab[int(i)] for i in encoded_sentence)
print(decoded_sentence)

i write rewrite and [UNK] rewrite again


In [14]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
import os
dataset_zip = "/content/drive/MyDrive/datasets/aclImdb_v1.tar.gz"
os.path.exists(dataset_zip)

True

In [16]:
!tar -xf /content/drive/MyDrive/datasets/aclImdb_v1.tar.gz

In [17]:
!rm -r /content/aclImdb/train/unsup

In [18]:
!cat /content/aclImdb/train/pos/0_9.txt

Bromwell High is a cartoon comedy. It ran at the same time as some other programs about school life, such as "Teachers". My 35 years in the teaching profession lead me to believe that Bromwell High's satire is much closer to reality than is "Teachers". The scramble to survive financially, the insightful students who can see right through their pathetic teachers' pomp, the pettiness of the whole situation, all remind me of the schools I knew and their students. When I saw the episode in which a student repeatedly tried to burn down the school, I immediately recalled ......... at .......... High. A classic line: INSPECTOR: I'm here to sack one of your teachers. STUDENT: Welcome to Bromwell High. I expect that many adults of my age think that Bromwell High is far fetched. What a pity that it isn't!

In [19]:
!cat /content/aclImdb/train/pos/10000_8.txt

Homelessness (or Houselessness as George Carlin stated) has been an issue for years but never a plan to help those on the street that were once considered human who did everything from going to school, work, or vote for the matter. Most people think of the homeless as just a lost cause while worrying about things such as racism, the war on Iraq, pressuring kids to succeed, technology, the elections, inflation, or worrying if they'll be next to end up on the streets.<br /><br />But what if you were given a bet to live on the streets for a month without the luxuries you once had from a home, the entertainment sets, a bathroom, pictures on the wall, a computer, and everything you once treasure to see what it's like to be homeless? That is Goddard Bolt's lesson.<br /><br />Mel Brooks (who directs) who stars as Bolt plays a rich man who has everything in the world until deciding to make a bet with a sissy rival (Jeffery Tambor) to see if he can live in the streets for thirty days without th

In [20]:
!cat /content/aclImdb/train/neg/10000_4.txt

Airport '77 starts as a brand new luxury 747 plane is loaded up with valuable paintings & such belonging to rich businessman Philip Stevens (James Stewart) who is flying them & a bunch of VIP's to his estate in preparation of it being opened to the public as a museum, also on board is Stevens daughter Julie (Kathleen Quinlan) & her son. The luxury jetliner takes off as planned but mid-air the plane is hi-jacked by the co-pilot Chambers (Robert Foxworth) & his two accomplice's Banker (Monte Markham) & Wilson (Michael Pataki) who knock the passengers & crew out with sleeping gas, they plan to steal the valuable cargo & land on a disused plane strip on an isolated island but while making his descent Chambers almost hits an oil rig in the Ocean & loses control of the plane sending it crashing into the sea where it sinks to the bottom right bang in the middle of the Bermuda Triangle. With air in short supply, water leaking in & having flown over 200 miles off course the problems mount for t

In [21]:
!cat /content/aclImdb/train/neg/0_3.txt

Story of a man who has unnatural feelings for a pig. Starts out with a opening scene that is a terrific example of absurd comedy. A formal orchestra audience is turned into an insane, violent mob by the crazy chantings of it's singers. Unfortunately it stays absurd the WHOLE time with no general narrative eventually making it just too off putting. Even those from the era should be turned off. The cryptic dialogue would make Shakespeare seem easy to a third grader. On a technical level it's better than you might think with some good cinematography by future great Vilmos Zsigmond. Future stars Sally Kirkland and Frederic Forrest can be seen briefly.

In [22]:

import os
import pathlib
import shutil
import random

base_dir = pathlib.Path("/content/aclImdb")
val_dir = base_dir / "val"
train_dir = base_dir / "train"

for category in ("neg", "pos"):
    os.makedirs(val_dir / category)

    files = os.listdir(train_dir / category)

    random.Random(1337).shuffle(files)

    num_val_samples = int(0.2 * len(files))
    val_files = files[-num_val_samples:]

    for fname in val_files:
        shutil.move(
            train_dir / category / fname,
            val_dir / category / fname
        )


In [23]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import text_dataset_from_directory

In [24]:
batch_size = 32

train_dataset = text_dataset_from_directory(
    "/content/aclImdb/train",
    batch_size=batch_size
)
val_dataset = text_dataset_from_directory(
    "/content/aclImdb/val",
    batch_size=batch_size
)
test_dataset = text_dataset_from_directory(
    "/content/aclImdb/test",
    batch_size=batch_size
)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


In [25]:
for inputs, targets in train_dataset.take(1):
    print("inputs.shape:", inputs.shape)
    print("inputs.dtype:", inputs.dtype)

    print("targets.shape:", targets.shape)
    print("targets.dtype:", targets.dtype)

    print("inputs[0]:", inputs[0])
    print("targets[0]:", targets[0])

inputs.shape: (32,)
inputs.dtype: <dtype: 'string'>
targets.shape: (32,)
targets.dtype: <dtype: 'int32'>
inputs[0]: tf.Tensor(b"Ask yourself where she got the gun? Remember what she was taught about the mark's mindset when the con is over? The gun had blanks and it was provided to her from the very beginning.<br /><br />When the patient comes back at the end she was SUPPOSED to see him drive away in the red convertible and lead her to the gang splitting up her 80 thousand.<br /><br />The patient was in on the con from the beginning.<br /><br />Mantegna does not die in the end - the gun had blanks.<br /><br />There - enough spoilers for you there? This is why people are giving it such high ratings. It's extremely original because of the hidden ending and how it cons MOST of the audience.", shape=(), dtype=string)
targets[0]: tf.Tensor(1, shape=(), dtype=int32)


## Experiment 1 - Single words (unigrams) with binary encoding

In [26]:
text_vectorization = layers.TextVectorization(
    max_tokens=20000,
    output_mode="multi_hot"
)

In [27]:
text_only_train_ds = train_dataset.map(lambda x, y: x)
text_vectorization.adapt(text_only_train_ds)

In [28]:
binary_1gram_train_ds = train_dataset.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4
)
binary_1gram_val_ds = val_dataset.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4
)
binary_1gram_test_ds = test_dataset.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4
)

In [29]:
for inputs, targets in binary_1gram_train_ds.take(1):
    print("inputs.shape:", inputs.shape)
    print("inputs.dtype:", inputs.dtype)

    print("targets.shape:", targets.shape)
    print("targets.dtype:", targets.dtype)

inputs.shape: (32, 20000)
inputs.dtype: <dtype: 'int64'>
targets.shape: (32,)
targets.dtype: <dtype: 'int32'>


In [30]:
def get_model(max_tokens=20000, hidden_dim=16):
    inputs = keras.Input(shape=(max_tokens,))
    x = layers.Dense(hidden_dim, activation="relu")(inputs)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer="rmsprop",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [31]:
model = get_model()

callbacks = keras.callbacks.ModelCheckpoint(
    "binary_1gram.keras",
    save_best_only=True
)

model.fit(
    binary_1gram_train_ds.cache(),
    validation_data=binary_1gram_val_ds.cache(),
    epochs=10,
    callbacks=[callbacks]
)

model = keras.models.load_model("binary_1gram.keras")

print(f"Test acc: {model.evaluate(binary_1gram_test_ds)[1]:.3f}")

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 13s 20ms/step - accuracy: 0.8283 - loss: 0.4052 - val_accuracy: 0.8912 - val_loss: 0.2794
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.8999 - loss: 0.2696 - val_accuracy: 0.8936 - val_loss: 0.2721
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.9169 - loss: 0.2348 - val_accuracy: 0.8926 - val_loss: 0.2868
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.9281 - loss: 0.2154 - val_accuracy: 0.8930 - val_loss: 0.3021
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.9313 - loss: 0.2076 - val_accuracy: 0.8928 - val_loss: 0.3158
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.9343 - loss: 0.2045 - val_accuracy: 0.8910 - val_loss: 0.3273
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.9354 - loss: 0.2033 - val_accuracy: 0.8904 - val_loss: 0.3447
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9384 - loss: 0.1942 - val_accur

- The model achieved an accuracy of 88.6% on the test data

## Bigrams with binary encoding

In [32]:
text_vectorization = layers.TextVectorization(
    ngrams=2,
    max_tokens=20000,
    output_mode="multi_hot"
)

In [33]:
text_vectorization.adapt(text_only_train_ds)

binary_2gram_train_ds = train_dataset.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4
)
binary_2gram_val_ds = val_dataset.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4
)
binary_2gram_test_ds = test_dataset.map(
    lambda x, y: (text_vectorization(x), y),
)

In [34]:
model = get_model()

callbacks = keras.callbacks.ModelCheckpoint(
    "binary_2gram.keras",
    save_best_only=True
)

model.fit(
    binary_2gram_train_ds.cache(),
    validation_data=binary_2gram_val_ds.cache(),
    epochs=10,
    callbacks=callbacks
)

model = keras.models.load_model("binary_2gram.keras")

print(f"Test acc: {model.evaluate(binary_2gram_test_ds)[1]:.3f}")

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 12s 18ms/step - accuracy: 0.8371 - loss: 0.3936 - val_accuracy: 0.9008 - val_loss: 0.2642
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.9093 - loss: 0.2586 - val_accuracy: 0.9034 - val_loss: 0.2654
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9262 - loss: 0.2218 - val_accuracy: 0.9042 - val_loss: 0.2860
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9328 - loss: 0.2006 - val_accuracy: 0.9006 - val_loss: 0.2959
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.9409 - loss: 0.1900 - val_accuracy: 0.8996 - val_loss: 0.3170
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9453 - loss: 0.1892 - val_accuracy: 0.8968 - val_loss: 0.3279
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9478 - loss: 0.1742 - val_accuracy: 0.8972 - val_loss: 0.3551
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.9480 - loss: 0.1806 - val_accurac

- The model acheived an accuracy of 89.2% (0.6% increase) accuracy on the test data

## Bigrams with TF-IDF encoding

In [35]:
text_vectorization = layers.TextVectorization(
    ngrams=2,
    max_tokens=20000,
    output_mode="tf_idf"
)

In [36]:
text_vectorization.adapt(text_only_train_ds)

tfidf_2gram_train_ds = train_dataset.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4
)
tfidf_2gram_val_ds = val_dataset.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4
)
tfidf_2gram_test_ds = test_dataset.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4
)

In [37]:
model = get_model()

callbacks = keras.callbacks.ModelCheckpoint(
    "tfidf_2gram.keras",
    save_best_only=True
)

model.fit(
    tfidf_2gram_train_ds.cache(),
    validation_data=tfidf_2gram_val_ds.cache(),
    epochs=10,
    callbacks=callbacks
)

model = keras.models.load_model("tfidf_2gram.keras")

print(f"Test acc: {model.evaluate(tfidf_2gram_test_ds)[1]:.3f}")

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 18s 28ms/step - accuracy: 0.7463 - loss: 0.4989 - val_accuracy: 0.8834 - val_loss: 0.3285
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8426 - loss: 0.3541 - val_accuracy: 0.8830 - val_loss: 0.2959
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.8546 - loss: 0.3273 - val_accuracy: 0.8864 - val_loss: 0.3024
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.8622 - loss: 0.3085 - val_accuracy: 0.8754 - val_loss: 0.3292
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.8691 - loss: 0.2913 - val_accuracy: 0.8840 - val_loss: 0.3252
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.8783 - loss: 0.2746 - val_accuracy: 0.8894 - val_loss: 0.3378
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.8899 - loss: 0.2527 - val_accuracy: 0.8854 - val_loss: 0.3643
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.8967 - loss: 0.2423 - val_accurac

- The model achieved 87.9 % accuracy (-1.3% decrease compared to binary_2grams, -0.7% decrease compared to binary_1gram)  on the test data

*Note*: An important machine-learning lesson: a more sophisticated feature representation does not necessarily lead to better performance on every dataset

## Exporting a model that accepts raw data

In [38]:
inputs = keras.Input(shape=(1,), dtype="string")

processed_inputs = text_vectorization(inputs)
outputs = model(processed_inputs)
inference_model = keras.Model(inputs, outputs)

In [39]:
raw_text_data = tf.convert_to_tensor([
    ["That was an axcellent movie, I loved it."]
])

predictions = inference_model(raw_text_data)

print(f"{float(predictions[0] * 100):.2f} percent positive")

71.16 percent positive
